Tutorial link: https://huggingface.co/learn/llm-course/chapter2/2

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00


### 1. Installing Dependencies
We first install the required libraries: `datasets` for loading data, `evaluate` for model evaluation, and `transformers[sentencepiece]` for working with transformer models and tokenization.

In [2]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9598046541213989},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

### 2. Using the High-Level Pipeline
Here we use the `pipeline` API from the `transformers` library for a quick, high-level abstraction. We instantiate a sentiment-analysis pipeline and pass two sentences to it. It automatically handles tokenization, model inference, and post-processing, giving us the predicted labels and scores.

In [3]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

### 3. Step 1: Tokenization
To understand what happens inside the pipeline, we first look at the tokenizer. `AutoTokenizer` automatically loads the correct tokenizer class for the given model checkpoint. Here, it loads the tokenizer for DistilBERT fine-tuned on the SST-2 dataset.

In [4]:
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


### 4. Processing Text with the Tokenizer
We pass the raw text to the tokenizer. Setting `padding=True` ensures all sequences have the same length by adding padding tokens. `truncation=True` ensures sequences don't exceed the model's maximum length. `return_tensors="pt"` returns PyTorch tensors.

The output contains `input_ids` (token IDs) and `attention_mask` (indicating which tokens are real and which are padding).

In [5]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 5. Step 2: Model Inference (Base Model)
We load the base model using `AutoModel`. This model architecture will only output hidden states (features) and does not include the specific task head (like sequence classification). Note the warning that some weights (from the classification head) are not used because we only loaded the base model.

In [6]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


### 6. Inspecting the Hidden States
We pass our tokenized inputs (tensors) into the base model using `**inputs` to unpack the dictionary. The model outputs hidden states.

The shape `[2, 16, 768]` corresponds to `[batch_size, sequence_length, hidden_size]`. This means we have 2 sentences, padded to 16 tokens each, and a 768-dimensional vector representing each token.

In [7]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

### 7. Step 2 (Continued): Model Inference with Classification Head
Instead of just the base model, we now load `AutoModelForSequenceClassification`. This includes the base model plus a sequence classification head on top. When we pass our inputs through this model, the output will be the raw logits for our classification task.

In [8]:
print(outputs.logits.shape)

torch.Size([2, 2])


### 8. Inspecting the Logits Shape
The shape of the logits is `[2, 2]`. This corresponds to `[batch_size, num_labels]`. We have 2 sentences and 2 possible classes (Positive or Negative).

In [9]:
print(outputs.logits)

tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)


### 9. Viewing the Raw Logits
The raw outputs from the model are called *logits*. These are unnormalized scores and do not represent probabilities, meaning they don't necessarily sum to 1 and can be negative.

In [10]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


### 10. Step 3: Post-Processing (Converting Logits to Probabilities)
To convert the raw logits into probabilities that sum to 1, we apply the Softmax activation function. Now we can see the probabilities for each class for both sentences.

In [11]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

### 11. Interpreting the Predictions
Finally, to interpret the probabilities, we can look at the model's configuration (`id2label`). It tells us that index 0 corresponds to 'NEGATIVE' and index 1 corresponds to 'POSITIVE'.

This matches the outputs from the high-level pipeline we ran at the beginning! We have now successfully reproduced the steps of the pipeline manually.